# 051 — group sites that share a structural design

Notebook 011 designs a 3-storey and a 5-storey EC8-gen2 DC2 CBF for every one of the 60
WP1 sites, driven only by the site's `S_alpha,475`. The design loads vary smoothly across
sites but the section catalogue is discrete, so many sites end up with a **byte-identical
structural design**. The FEMA P695 IDA wired up by 011 uses the same 22-record far-field
set at every site, so an IDA result depends *only* on the structure — running the same
design at eight sites is eight times the compute for one answer.

This notebook groups the sites whose designs are identical, nominates one representative
site per group, and writes the mapping so only the unique buildings need analysing.

## What "the same design" means

`config_structural_model.py` builds the OpenSees model from `json.load(f)["structure"]`
of the design file and nothing else. So *identical `structure` block ⇒ identical model ⇒
identical IDA*. That is the criterion used here: the whole `structure` block is compared,
not the period and not the section names. Two designs can share every section yet differ
in gusset thickness, effective dimensions or offsets — those are different models.

## Dependencies

Run notebook **011** first: it writes the per-site designs under
`casestudy_designs_site_specific` and the `site_designs_summary.csv` index.

## Outputs

- `unique_structural_designs.json` — nested: storey count → groups → member sites.
- `unique_structural_designs.csv` — one row per (site, storey) with its `group_id`,
  `representative_site` and `is_representative`, for joining in notebooks 014 / 060 / 061.

Both land in `data_processed/08_casestudy_structure_datasets/` (config keys
`unique_structural_designs` / `unique_structural_designs_csv`).

## 0. Setup & parameters

In [11]:
import json
import hashlib
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

from phd_project.config import config
from phd_project.scripts.case_study_design_scripts.design_file_helpers import load_design_file

cfg = config.load_config()

# --- parameters -----------------------------------------------------------
STOREYS = [3, 5]
DUCTILITY_CLASS = 2

# Rounding applied to every float in the `structure` block before hashing. None = exact
# comparison, which is the default: the designs come from a deterministic algorithm, so
# equal designs are bit-for-bit equal (rounding to 4 d.p. gives the same groups). The
# knob only exists in case a future re-design introduces numerical noise.
FLOAT_TOLERANCE = None

DESIGN_ROOT = Path(cfg["models"]["casestudy_designs_site_specific"])
SUMMARY_CSV = Path(cfg["models"]["site_specific_designs_summary"])
OUT_JSON = Path(cfg["proc_data"]["unique_structural_designs"])
OUT_CSV = Path(cfg["proc_data"]["unique_structural_designs_csv"])


def make_tag(n: int, ii: int) -> str:
    """Site building tag, matching notebook 011."""
    return f"{n}s_cbf_dc{DUCTILITY_CLASS}_site{ii}"


def design_json_path(n: int, ii: int) -> Path:
    tag = make_tag(n, ii)
    return DESIGN_ROOT / tag / f"{tag}_out.json"


print(f"DESIGN_ROOT = {DESIGN_ROOT}")

DESIGN_ROOT = C:\Users\clemettn\Documents\phd\casestudy_structures\concentrically_braced_frames\ec8_gen2_site_specific_designs


## 1. Discover sites and their design files

Sites come from `sites.csv` (notebook 001): the positional row index `ii` *is* the site
index, the convention used by the `site_{ii}` folders everywhere else in the project.

`site_designs_summary.csv` is the index of what notebook 011 actually managed to design.
Any (site, storey) whose design failed or whose `_out.json` is missing is **warned and
skipped** rather than crashing — it simply cannot be grouped.

In [12]:
sites_df = pd.read_csv(cfg["results"]["selected_sites_csv"])
all_sites = list(range(len(sites_df)))

summary_df = pd.read_csv(SUMMARY_CSV).set_index("tag")
failed_tags = set(summary_df.index[~summary_df["success"].astype(bool)])

# (storeys, site) -> design json path, only for designs that exist and succeeded
design_files: dict[tuple[int, int], Path] = {}
skipped: list[tuple[int, int, str]] = []

for n in STOREYS:
    for ii in all_sites:
        tag = make_tag(n, ii)
        path = design_json_path(n, ii)
        if tag not in summary_df.index:
            skipped.append((n, ii, "tag absent from site_designs_summary.csv"))
        elif tag in failed_tags:
            skipped.append((n, ii, "design flagged unsuccessful in site_designs_summary.csv"))
        elif not path.is_file():
            skipped.append((n, ii, f"{path.name} not found"))
        else:
            design_files[(n, ii)] = path

print(f"{len(all_sites)} sites x {len(STOREYS)} storey counts -> {len(design_files)} designs found")
for n, ii, why in skipped:
    print(f"WARNING: skipping site {ii} [{n}s]: {why}")

60 sites x 2 storey counts -> 120 designs found


## 2. Structure fingerprint

A stable hash of the `structure` block. Two designs with the same fingerprint build the
same OpenSees model, so one analysis covers both.

In [13]:
def _round(obj, nd):
    """Recursively round every float in a nested json structure."""
    if nd is None:
        return obj
    if isinstance(obj, dict):
        return {k: _round(v, nd) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_round(v, nd) for v in obj]
    if isinstance(obj, float):
        return round(obj, nd)
    return obj


def structure_fingerprint(structure: dict) -> str:
    """Stable hash of the model-defining payload of a design file.

    The OpenSees model is built from `json["structure"]` alone (see
    `config_structural_model.py`), so equal fingerprints mean an identical model and
    therefore an identical IDA.
    """
    canonical = json.dumps(_round(structure, FLOAT_TOLERANCE), sort_keys=True,
                           separators=(",", ":"))
    return hashlib.sha256(canonical.encode()).hexdigest()[:16]


def section_names(structure: dict) -> dict[str, list[str]]:
    """The section assigned to every column / beam / brace, for reporting."""
    return {kind: [e["section"] for e in structure[kind]]
            for kind in ("columns", "beams", "braces")}


structures = {key: load_design_file(path)["structure"] for key, path in design_files.items()}
fingerprints = {key: structure_fingerprint(s) for key, s in structures.items()}

print(f"fingerprinted {len(fingerprints)} designs (FLOAT_TOLERANCE = {FLOAT_TOLERANCE})")

fingerprinted 120 designs (FLOAT_TOLERANCE = None)


## 3. Group the sites

One group per distinct fingerprint, within a storey count. Groups are numbered by their
lowest member site so the ids are stable across re-runs, and the **representative site is
the lowest site index in the group** — the one to actually analyse.

In [14]:
# storeys -> list of group dicts, ordered by lowest member site
groups: dict[int, list[dict]] = {}

for n in STOREYS:
    by_fingerprint: dict[str, list[int]] = defaultdict(list)
    for (nn, ii), fp in fingerprints.items():
        if nn == n:
            by_fingerprint[fp].append(ii)

    ordered = sorted(by_fingerprint.items(), key=lambda kv: min(kv[1]))
    groups[n] = [
        {
            "group_id": gid,
            "fingerprint": fp,
            "representative_site": min(sites),
            "representative_tag": make_tag(n, min(sites)),
            "sites": sorted(sites),
            "n_sites": len(sites),
            "sections": section_names(structures[(n, min(sites))]),
        }
        for gid, (fp, sites) in enumerate(ordered)
    ]

for n in STOREYS:
    print(f"{n}s: {sum(g['n_sites'] for g in groups[n])} sites -> {len(groups[n])} unique designs")

3s: 60 sites -> 25 unique designs
5s: 60 sites -> 26 unique designs


## 4. Report & checks

Group sizes, the groups themselves, and the compute saving. The consistency checks are
cheap and catch a broken fingerprint: every design must land in exactly one group, each
representative must belong to its own group, and every member of a group must share the
design period `T` recorded independently in `site_designs_summary.csv`.

In [15]:
n_total = len(design_files)
n_unique = sum(len(groups[n]) for n in STOREYS)

for n in STOREYS:
    sizes = Counter(g["n_sites"] for g in groups[n])
    print(f"\n--- {n}s: {len(groups[n])} unique designs ---")
    print("  group sizes: " + ", ".join(f"{size} site(s) x{cnt}"
                                        for size, cnt in sorted(sizes.items(), reverse=True)))
    for g in groups[n]:
        shared = "" if g["n_sites"] == 1 else f"   <- {g['n_sites']} sites share this design"
        print(f"  [{g['group_id']:>2}] rep site {g['representative_site']:>2}: "
              f"{g['sites']}{shared}")

print(f"\nAnalyses needed: {n_unique} instead of {n_total} "
      f"({n_total - n_unique} fewer, {1 - n_unique / n_total:.0%} saved)")


--- 3s: 25 unique designs ---
  group sizes: 8 site(s) x1, 6 site(s) x1, 5 site(s) x1, 4 site(s) x2, 3 site(s) x4, 2 site(s) x5, 1 site(s) x11
  [ 0] rep site  0: [0, 4, 8, 19, 25, 26, 27, 29]   <- 8 sites share this design
  [ 1] rep site  1: [1, 2, 5, 6, 16, 23]   <- 6 sites share this design
  [ 2] rep site  3: [3]
  [ 3] rep site  7: [7, 10, 11, 18, 24]   <- 5 sites share this design
  [ 4] rep site  9: [9, 20, 22]   <- 3 sites share this design
  [ 5] rep site 12: [12]
  [ 6] rep site 13: [13, 28]   <- 2 sites share this design
  [ 7] rep site 14: [14]
  [ 8] rep site 15: [15]
  [ 9] rep site 17: [17, 21]   <- 2 sites share this design
  [10] rep site 30: [30]
  [11] rep site 31: [31, 34, 43, 59]   <- 4 sites share this design
  [12] rep site 32: [32, 33, 36, 50]   <- 4 sites share this design
  [13] rep site 35: [35, 39, 58]   <- 3 sites share this design
  [14] rep site 37: [37]
  [15] rep site 38: [38]
  [16] rep site 40: [40]
  [17] rep site 41: [41, 54, 57]   <- 3 sites shar

In [16]:
# --- consistency checks ---------------------------------------------------
for n in STOREYS:
    members = sorted(ii for g in groups[n] for ii in g["sites"])
    expected = sorted(ii for (nn, ii) in design_files if nn == n)
    assert members == expected, f"{n}s: sites missing from / duplicated across groups"
    for g in groups[n]:
        assert g["representative_site"] in g["sites"]

# every member of a group should share the design period recorded by nb 011
mismatches = []
for n in STOREYS:
    for g in groups[n]:
        periods = {round(float(summary_df.loc[make_tag(n, ii), "T"]), 6) for ii in g["sites"]}
        if len(periods) > 1:
            mismatches.append((n, g["group_id"], sorted(periods)))

if mismatches:
    for n, gid, periods in mismatches:
        print(f"WARNING: {n}s group {gid} spans several design periods {periods} - "
              f"the fingerprint and site_designs_summary.csv disagree")
else:
    print("checks passed: every site grouped once, representatives valid, "
          "design periods consistent within every group")

checks passed: every site grouped once, representatives valid, design periods consistent within every group


## 5. Write the outputs

The JSON is the readable description of the groups; the CSV is the flat per-(site, storey)
join table for downstream notebooks.

In [17]:
OUT_JSON.parent.mkdir(parents=True, exist_ok=True)

payload = {
    "generated_by": "notebooks/03_WP1_ground_motion_set/051-group_sites_by_structural_design.ipynb",
    "design_root": DESIGN_ROOT.as_posix(),
    "match_criterion": "exact json['structure'] block of {tag}_out.json",
    "float_tolerance": FLOAT_TOLERANCE,
    "groups": {str(n): groups[n] for n in STOREYS},
    "summary": {
        str(n): {
            "n_sites": sum(g["n_sites"] for g in groups[n]),
            "n_unique": len(groups[n]),
        }
        for n in STOREYS
    },
}

with open(OUT_JSON, "w") as f:
    json.dump(payload, f, indent=2)

rows = [
    {
        "storeys": n,
        "site": ii,
        "tag": make_tag(n, ii),
        "group_id": g["group_id"],
        "fingerprint": g["fingerprint"],
        "representative_site": g["representative_site"],
        "representative_tag": g["representative_tag"],
        "is_representative": ii == g["representative_site"],
        "n_sites_in_group": g["n_sites"],
    }
    for n in STOREYS
    for g in groups[n]
    for ii in g["sites"]
]
groups_df = pd.DataFrame(rows).sort_values(["storeys", "site"]).reset_index(drop=True)
groups_df.to_csv(OUT_CSV, index=False)

print(f"wrote {OUT_JSON}")
print(f"wrote {OUT_CSV}  ({len(groups_df)} rows, "
      f"{int(groups_df['is_representative'].sum())} representatives)")
groups_df.head(10)

wrote C:\Users\clemettn\Documents\phd\data_processed\08_casestudy_structure_datasets\unique_structural_designs.json
wrote C:\Users\clemettn\Documents\phd\data_processed\08_casestudy_structure_datasets\unique_structural_designs.csv  (120 rows, 51 representatives)


,storeys,site,tag,group_id,fingerprint,representative_site,representative_tag,is_representative,n_sites_in_group
0,3,0,3s_cbf_dc2_site0,0,26435b8268ee6bbf,0,3s_cbf_dc2_site0,True,8
1,3,1,3s_cbf_dc2_site1,1,f401ac831c94d559,1,3s_cbf_dc2_site1,True,6
2,3,2,3s_cbf_dc2_site2,1,f401ac831c94d559,1,3s_cbf_dc2_site1,False,6
3,3,3,3s_cbf_dc2_site3,2,84d5e0674089226d,3,3s_cbf_dc2_site3,True,1
4,3,4,3s_cbf_dc2_site4,0,26435b8268ee6bbf,0,3s_cbf_dc2_site0,False,8
5,3,5,3s_cbf_dc2_site5,1,f401ac831c94d559,1,3s_cbf_dc2_site1,False,6
6,3,6,3s_cbf_dc2_site6,1,f401ac831c94d559,1,3s_cbf_dc2_site1,False,6
7,3,7,3s_cbf_dc2_site7,3,264a066dac41a903,7,3s_cbf_dc2_site7,True,5
8,3,8,3s_cbf_dc2_site8,0,26435b8268ee6bbf,0,3s_cbf_dc2_site0,False,8
9,3,9,3s_cbf_dc2_site9,4,7ebd08cb5f289c87,9,3s_cbf_dc2_site9,True,3


## 6. Reading the groups back

`load_design_groups` lives in `phd_project.scripts.WP1_ground_motion_set.design_groups`
so any notebook can use it (alongside `representative_sites` / `representative_of`).
Filter on `is_representative` to get the buildings that actually need analysing, then map
results back to the other sites via `group_id`.

In [18]:
from phd_project.scripts.WP1_ground_motion_set.design_groups import (
    load_design_groups,
    representative_sites,
)

check = load_design_groups(cfg)
print(f"{len(check)} (site, storey) pairs, "
      f"{int(check['is_representative'].sum())} to analyse:")
for n in STOREYS:
    print(f"  {n}s: {representative_sites(n, groups_df=check)}")

120 (site, storey) pairs, 51 to analyse:
  3s: [0, 1, 3, 7, 9, 12, 13, 14, 15, 17, 30, 31, 32, 35, 37, 38, 40, 41, 42, 44, 45, 47, 48, 53, 55]
  5s: [0, 7, 10, 11, 17, 21, 25, 30, 31, 32, 33, 34, 37, 38, 39, 40, 41, 42, 44, 45, 48, 49, 50, 51, 58, 59]
